In [33]:
import pandas as pd

In [34]:
from google.colab import files
uploaded = files.upload()

Saving data.csv to data (1).csv


In [35]:
df = pd.read_csv('data.csv', encoding='latin1')
df.head()

,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,12/1/2010 8:26,2.55,17850.0,United Kingdom
1,536365,71053,WHITE METAL LANTERN,6,12/1/2010 8:26,3.39,17850.0,United Kingdom
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,12/1/2010 8:26,2.75,17850.0,United Kingdom
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,12/1/2010 8:26,3.39,17850.0,United Kingdom
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,12/1/2010 8:26,3.39,17850.0,United Kingdom


In [36]:
#size of dataset
df.shape

(541909, 8)

In [37]:
#type of data
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 541909 entries, 0 to 541908
Data columns (total 8 columns):
 #   Column       Non-Null Count   Dtype  
---  ------       --------------   -----  
 0   InvoiceNo    541909 non-null  object 
 1   StockCode    541909 non-null  object 
 2   Description  540455 non-null  object 
 3   Quantity     541909 non-null  int64  
 4   InvoiceDate  541909 non-null  object 
 5   UnitPrice    541909 non-null  float64
 6   CustomerID   406829 non-null  float64
 7   Country      541909 non-null  object 
dtypes: float64(2), int64(1), object(5)
memory usage: 33.1+ MB


In [38]:
#basic ststistic
df.describe()

,Quantity,UnitPrice,CustomerID
count,541909.000000,541909.000000,406829.000000
mean,9.552250,4.611114,15287.690570
std,218.081158,96.759853,1713.600303
min,-80995.000000,-11062.060000,12346.000000
25%,1.000000,1.250000,13953.000000
50%,3.000000,2.080000,15152.000000
75%,10.000000,4.130000,16791.000000
max,80995.000000,38970.000000,18287.000000


Очистка даних

In [39]:
#Крок 1 — видаляємо рядки без CustomerID:
df = df.dropna(subset=['CustomerID'])
print(df.shape)

(406829, 8)


In [40]:
#Крок 2 — видаляємо повернення і нульові ціни:
df = df[df['Quantity'] > 0]
df = df[df['UnitPrice'] > 0]
print(df.shape)

(397884, 8)


In [41]:
#Крок 3 — виправляємо типи:
df['InvoiceDate'] = pd.to_datetime(df['InvoiceDate'])
df['CustomerID'] = df['CustomerID'].astype(int)
print(df.dtypes)

InvoiceNo              object
StockCode              object
Description            object
Quantity                int64
InvoiceDate    datetime64[ns]
UnitPrice             float64
CustomerID              int64
Country                object
dtype: object


Метрики

In [42]:
df['Revenue']=df['Quantity']*df['UnitPrice']

In [43]:
df['Revenue'].sum()

np.float64(8911407.904)

In [44]:
#Кількість унікальних замовлень:
df['InvoiceNo'].nunique()

18532

In [45]:
#Кількість унікальних клієнтів:
df['CustomerID'].nunique()

4338

In [46]:
#AOV
order_revenue = df.groupby('InvoiceNo')['Revenue'].sum()
aov = order_revenue.mean()
print(f'AOV: £{aov:.2f}')

AOV: £480.87


In [47]:
#Які місяці були найприбутковішими?
df['Month'] = df['InvoiceDate'].dt.to_period('M')
monthly_revenue = df.groupby('Month')['Revenue'].sum().sort_values(ascending=False)
print(monthly_revenue)

Month
2011-11    1161817.380
2011-10    1039318.790
2011-09     952838.382
2011-05     678594.560
2011-06     661213.690
2011-08     645343.900
2011-07     600091.011
2011-03     595500.760
2010-12     572713.890
2011-01     569445.040
2011-12     518192.790
2011-04     469200.361
2011-02     447137.350
Freq: M, Name: Revenue, dtype: float64


Це класична сезонність для e-commerce — осінь різко зростає бо люди готуються до Різдва, купують подарунки. Листопад — це також Black Friday.
Грудень 2011 виглядає низьким — але датасет, швидше за все, просто не містить повний місяць, тому порівняння некоректне.

In [48]:
#top-products
top_products = df.groupby('Description')['Revenue'].sum()
top_products = top_products.sort_values(ascending=False).head(10)
print(top_products)

Description
PAPER CRAFT , LITTLE BIRDIE           168469.60
REGENCY CAKESTAND 3 TIER              142592.95
WHITE HANGING HEART T-LIGHT HOLDER    100448.15
JUMBO BAG RED RETROSPOT                85220.78
MEDIUM CERAMIC TOP STORAGE JAR         81416.73
POSTAGE                                77803.96
PARTY BUNTING                          68844.33
ASSORTED COLOUR BIRD ORNAMENT          56580.34
Manual                                 53779.93
RABBIT NIGHT LIGHT                     51346.20
Name: Revenue, dtype: float64


Топ товари за виручкою:
Явні лідери:

Paper Craft Little Birdie — £168,469 👑
Regency Cakestand 3 Tier — £142,592
White Hanging Heart T-Light Holder — £100,448

⚠️ Підозрілі рядки:

POSTAGE — £77,803 — це не товар, це вартість доставки
Manual — £53,779 — теж не товар, схоже на ручні коригування


💡 Бізнес-висновок:
Магазин продає декор та подарунки — свічники, підставки для тортів, прикраси. Це типовий британський gift shop. Ця інформація важлива — вона пояснює чому листопад такий сильний (подарунки до Різдва).
Потрібно  очистити перед фінальним звітом:

In [49]:
df = df[~df['Description'].isin(['POSTAGE', 'Manual'])]

In [50]:
#top-customers
top_customers = df.groupby('CustomerID')['Revenue'].sum()
top_customers = top_customers.sort_values(ascending=False).head(10)
print(top_customers)

CustomerID
14646    279138.02
18102    259657.30
17450    194550.79
16446    168472.50
14911    140450.72
12415    124564.53
14156    117379.63
17511     91062.38
12346     77183.60
16029     72882.09
Name: Revenue, dtype: float64


Бізнес-висновок:Топ-1 клієнт приніс £279,138 — це 3.1% від усієї виручки магазину сам по собі. Топ-10 клієнтів разом — це приблизно £1.7 млн або ~19% виручки.Це класичний принцип Парето (80/20) в дії — невелика кількість клієнтів генерує велику частину доходу. Для бізнесу це сигнал: цих клієнтів треба берегти особливо.

На наступному кроці зробимо аналіз ще глибшим. У нас є дані по клієнтах — подивимось хто купує повторно, а хто зробив лише одне замовлення.Це важлива метрика для e-commerce — називається Retention (утримання клієнтів).

In [51]:
orders_per_customer = df.groupby('CustomerID')['InvoiceNo'].nunique()
print(orders_per_customer.value_counts().head(10))

InvoiceNo
1     1505
2      830
3      504
4      394
5      237
6      173
7      138
8       98
9       69
10      55
Name: count, dtype: int64


In [52]:
#який відсоток клієнтів купив лише один раз:
one_time = (orders_per_customer == 1).sum()
total = orders_per_customer.count()
print(f'Одноразові клієнти: {one_time}')
print(f'Всього клієнтів: {total}')
print(f'Відсоток: {one_time/total*100:.1f}%')

Одноразові клієнти: 1505
Всього клієнтів: 4335
Відсоток: 34.7%


RFM аналіз. Це один з найвідоміших інструментів в e-commerce аналітиці.
RFM:

Recency — коли востаннє купував клієнт
Frequency — як часто купує
Monetary — скільки витратив загалом

Це дозволяє сегментувати клієнтів — наприклад знайти "VIP клієнтів" або "клієнтів що засинають".

In [53]:
import datetime

# Визначаємо дату відліку
snapshot_date = df['InvoiceDate'].max() + datetime.timedelta(days=1)

rfm = df.groupby('CustomerID').agg(
    Recency   = ('InvoiceDate', lambda x: (snapshot_date - x.max()).days),
    Frequency = ('InvoiceNo', 'nunique'),
    Monetary  = ('Revenue', 'sum')
).reset_index()

print(rfm.head(10))

   CustomerID  Recency  Frequency  Monetary
0       12346      326          1  77183.60
1       12347        2          7   4310.00
2       12348       75          4   1437.24
3       12349       19          1   1457.55
4       12350      310          1    294.40
5       12352       36          7   1385.74
6       12353      204          1     89.00
7       12354      232          1   1079.40
8       12355      214          1    459.40
9       12356       23          3   2487.43


RFM таблиця готова.

📊 Читаємо таблицю:
КлієнтRecencyFrequencyMonetaryВисновок
123472 дні7 замовлень£4,310🌟 VIP — активний і лояльний12346326 днів1 замовлення£77,183😴 Купив багато але давно зник12353204 дні1 замовлення£89❌ Втрачений клієнт

In [54]:
#Тепер присвоїмо кожному клієнту оцінки від 1 до 4 по кожній метриці — це і є RFM скоринг:
rfm['R_score'] = pd.qcut(rfm['Recency'],   q=4, labels=[4,3,2,1])
rfm['F_score'] = pd.qcut(rfm['Frequency'].rank(method='first'), q=4, labels=[1,2,3,4])
rfm['M_score'] = pd.qcut(rfm['Monetary'],  q=4, labels=[1,2,3,4])

rfm['RFM_Score'] = rfm['R_score'].astype(str) + rfm['F_score'].astype(str) + rfm['M_score'].astype(str)

print(rfm.head(10))

   CustomerID  Recency  Frequency  Monetary R_score F_score M_score RFM_Score
0       12346      326          1  77183.60       1       1       4       114
1       12347        2          7   4310.00       4       4       4       444
2       12348       75          4   1437.24       2       3       3       233
3       12349       19          1   1457.55       3       1       3       313
4       12350      310          1    294.40       1       1       1       111
5       12352       36          7   1385.74       3       4       3       343
6       12353      204          1     89.00       1       1       1       111
7       12354      232          1   1079.40       1       1       3       113
8       12355      214          1    459.40       1       1       2       112
9       12356       23          3   2487.43       3       3       4       334


In [55]:
def segment(row):
    score = row['RFM_Score']
    r = int(score[0])
    f = int(score[1])
    m = int(score[2])

    if r >= 3 and f >= 3 and m >= 3:
        return 'VIP'
    elif r >= 3 and f >= 2:
        return 'Лояльний'
    elif r >= 3 and f == 1:
        return 'Новий'
    elif r == 2:
        return 'Засинає'
    else:
        return 'Втрачений'

rfm['Segment'] = rfm.apply(segment, axis=1)

print(rfm['Segment'].value_counts())

Segment
VIP          1318
Втрачений    1079
Засинає      1074
Лояльний      605
Новий         259
Name: count, dtype: int64


1318 VIP клієнтів — це золотий актив магазину. А 2153 клієнти (втрачені + засинають) — це потенційна виручка яку магазин ризикує втратити. Якби повернути хоча б половину — це мільйони фунтів.

In [56]:
# Об'єднуємо всі дані в один файл
df_full = df.merge(rfm[['CustomerID', 'Recency', 'Frequency', 'Monetary', 'Segment']],
                   on='CustomerID',
                   how='left')

df_full['Month'] = df_full['InvoiceDate'].dt.to_period('M').astype(str)

df_full.to_csv('ecommerce_full.csv', index=False)
files.download('ecommerce_full.csv')

print(df_full.shape)
print(df_full.columns.tolist())

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

(396501, 14)
['InvoiceNo', 'StockCode', 'Description', 'Quantity', 'InvoiceDate', 'UnitPrice', 'CustomerID', 'Country', 'Revenue', 'Month', 'Recency', 'Frequency', 'Monetary', 'Segment']
